# channel-analytics 30 分钟上手

本 notebook 演示:
1. 启动 Pipeline
2. 跑 ETL(空数据)
3. 加载示例品牌数据
4. 跑带品牌的 ETL
5. 查看 RPT 结果

**前置**: 已 `pip install -e .[dev]`,已 `cp .env.example .env && channel-analytics init-secrets`。

In [ ]:
from channel_analytics.etl.pipeline import run_default_etl
from channel_analytics.etl.providers.example_minimal import ExampleMinimalProvider
import pandas as pd

# 1. 空数据跑一遍(验证 pipeline 跑通)
ctx = run_default_etl({}, brand_provider_dotted="channel_analytics.etl.providers.example_minimal:ExampleMinimalProvider")
print(f"STG: {len(ctx.stg)}, RPT: {len(ctx.rpt)}")

In [ ]:
# 2. 加载示例品牌(占位,严禁写真实品牌名)
from channel_analytics.etl.brand import BrandWhitelistProvider

class DemoBrandProvider(BrandWhitelistProvider):
    def get_brands(self):
        return frozenset({"Brand A", "Brand B", "Brand C"})

demo = DemoBrandProvider()
print(f"Demo brand whitelist: {demo.get_brands()}")

In [ ]:
# 3. 跑带示例数据的 ETL
raw = {
    "stg_stock_current": pd.DataFrame({
        "warehouse": ["W1", "W1", "W1"],
        "material_code": ["MAT-001", "MAT-001", "MAT-002"],
        "material_name": ["示例物料001", "示例物料001", "示例物料002"],
        "batch_number": ["B001", "B002", "B003"],
        "brand": ["Brand A", "Brand A", "Brand B"],
        "available_qty": [100, 50, 200],
        "expiry_date": pd.to_datetime(["2028-01-01", "2026-06-01", "2027-12-31"]),
    }),
    "stg_sales_out": pd.DataFrame({
        "material_code": ["MAT-001", "MAT-002"],
        "batch_number": ["B001", "B003"],
        "doc_date": pd.to_datetime(["2026-05-01", "2026-05-15"]),
        "sales_out_qty": [30, 80],
    }),
    "stg_purchase_req": pd.DataFrame(columns=["order_no", "material_code", "order_qty"]),
    "stg_purchase_order": pd.DataFrame(columns=["order_no", "material_code", "brand_sales_no"]),
    "stg_stock_in": pd.DataFrame(columns=["brand_sales_no", "material_code", "stock_in_qty"]),
}

ctx = run_default_etl(raw, brand_provider_dotted="channel_analytics.etl.providers.example_minimal:ExampleMinimalProvider")
print(f"STG: {list(ctx.stg)}, RPT: {list(ctx.rpt)}")

In [ ]:
# 4. 查看 RPT 结果
if "rpt_expiry_warning" in ctx.rpt:
    display(ctx.rpt["rpt_expiry_warning"])
if "rpt_turnover_warning" in ctx.rpt:
    display(ctx.rpt["rpt_turnover_warning"])